# 03 — Results Comparison

Compare all four segmentation models:
- Quantitative metrics table (IoU, Dice, Precision, Recall)
- Bar charts per metric
- Qualitative side-by-side prediction visualisation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results_csv = '../outputs/results/metrics_summary.csv'
df = pd.read_csv(results_csv, index_col='model')
print(df.to_string())

In [ ]:
# Bar chart comparison
metrics = ['iou', 'dice', 'precision', 'recall']
labels  = ['IoU', 'Dice', 'Precision', 'Recall']

x = np.arange(len(df))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, (m, label) in enumerate(zip(metrics, labels)):
    ax.bar(x + i * width, df[m], width, label=label)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df.index, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Crack Segmentation Metrics')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Qualitative comparison — first 3 test images
from pathlib import Path
from PIL import Image

MODELS = ['deeplabv3plus', 'ppliteseg', 'pidnet', 'maskrcnn']
pred_root = Path('../outputs/predictions')
gt_root   = Path('../concreteCrackSegmentationDataset/BW')
rgb_root  = Path('../concreteCrackSegmentationDataset/rgb')

with open('../data/splits/test.txt') as f:
    test_stems = [l.strip() for l in f][:3]  # first 3

rgb_index = {p.stem.lower(): p for p in rgb_root.iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}

for stem in test_stems:
    n_cols = 2 + len(MODELS)  # RGB + GT + 4 models
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    rgb_p = rgb_index.get(stem.lower())
    axes[0].imshow(Image.open(rgb_p))
    axes[0].set_title('RGB')

    gt_p = gt_root / f'{stem}.jpg'
    axes[1].imshow(Image.open(gt_p), cmap='gray')
    axes[1].set_title('Ground Truth')

    for j, model in enumerate(MODELS):
        pred_p = pred_root / model / f'{stem}.png'
        if pred_p.exists():
            axes[2 + j].imshow(Image.open(pred_p), cmap='gray')
        axes[2 + j].set_title(model)

    for ax in axes:
        ax.axis('off')
    plt.suptitle(stem, fontsize=10)
    plt.tight_layout()
    plt.show()